# Data Preparation: Clean Data -> Feature -> Format

Notebook này viết lại gọn hơn bản cũ. Mục tiêu là chuẩn bị dữ liệu cho bài toán daily `Revenue`/`COGS` forecasting, không làm lại EDA rộng.

Luồng chính:

1. **Clean Data**: load, correct type, duplicate, missing, logic check, outlier + biểu đồ/bảng kiểm tra.
2. **Feature**: tạo daily table, sinh feature mới, kiểm tra và chọn feature dùng được.
3. **Format & Export**: split train/valid theo thời gian, impute/scale bằng train-only, PCA nếu bật, export CSV sạch.


## 0. Setup ngắn

Chỉ khai báo config, load thư viện, và helper nhỏ dùng lại.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.tseries.holiday import USFederalHolidayCalendar

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

DATA_DIR = Path("data")
OUT_DIR = Path("outputs/data_preparation_clean_feature_format")
REPORT_DIR = OUT_DIR / "reports"
CLEAN_DIR = OUT_DIR / "clean_tables"
for folder in [OUT_DIR, REPORT_DIR, CLEAN_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TARGETS = ["Revenue", "COGS"]
VALID_START_DATE = pd.Timestamp("2021-01-01")
LAGS = [1, 7, 14, 28, 365]
ROLL_WINDOWS = [7, 28, 90]
APPLY_PCA = False

FILES = {
    "sales": "sales.csv",
    "sample_submission": "sample_submission.csv",
    "web_traffic": "web_traffic.csv",
    "orders": "orders.csv",
    "order_items": "order_items.csv",
    "products": "products.csv",
    "returns": "returns.csv",
    "reviews": "reviews.csv",
    "promotions": "promotions.csv",
    "inventory": "inventory.csv",
    "customers": "customers.csv",
}

print("Output:", OUT_DIR.resolve())


In [ ]:
def has_cols(df, cols):
    return all(c in df.columns for c in cols)

def show(df, n=5):
    display(df.head(n))

def iqr_bounds(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

def barh(df, value_col, label_col, title, color="#4C78A8", figsize=(8, 4)):
    if len(df) == 0:
        print("No data to plot:", title)
        return
    fig, ax = plt.subplots(figsize=figsize)
    ax.barh(df[label_col], df[value_col], color=color)
    ax.set_title(title)
    ax.set_xlabel(value_col)
    plt.tight_layout()
    plt.show()

def add_lag_roll(df, date_col, cols, lags=LAGS, windows=ROLL_WINDOWS):
    df = df.sort_values(date_col).copy()
    made = []
    for col in cols:
        if col not in df.columns:
            continue
        for lag in lags:
            name = f"{col}_lag_{lag}d"
            df[name] = df[col].shift(lag)
            made.append(name)
        past = df[col].shift(1)
        for window in windows:
            name = f"{col}_rolling_{window}d_mean"
            df[name] = past.rolling(window, min_periods=1).mean()
            made.append(name)
    return df, made


## 1. CLEAN DATA

Mục này làm mọi thứ thuộc làm sạch: load file, sửa type, duplicate, missing overview, logic check, outlier và bảng/biểu đồ để quyết định. Ở đây **không tự xóa outlier** nếu chưa có bằng chứng nghiệp vụ.


In [ ]:
# Load các CSV có trong data/. File phụ nếu thiếu thì chỉ mất feature liên quan.
raw = {}
load_rows = []
for name, file_name in FILES.items():
    path = DATA_DIR / file_name
    if not path.exists():
        load_rows.append([name, file_name, "missing", 0, 0])
        continue
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    raw[name] = df
    load_rows.append([name, file_name, "loaded", len(df), df.shape[1]])

load_report = pd.DataFrame(load_rows, columns=["table", "file", "status", "rows", "cols"])
display(load_report)


In [ ]:
# Correct date type.
clean = {name: df.copy() for name, df in raw.items()}

date_cols = {
    "sales": ["Date"],
    "sample_submission": ["Date"],
    "web_traffic": ["date"],
    "orders": ["order_date"],
    "returns": ["return_date"],
    "reviews": ["review_date"],
    "promotions": ["start_date", "end_date"],
    "inventory": ["snapshot_date"],
    "customers": ["signup_date"],
}
type_rows = []

for table, cols in date_cols.items():
    if table not in clean:
        continue
    for col in cols:
        if col not in clean[table].columns:
            continue
        before = clean[table][col].isna().sum()
        parsed = pd.to_datetime(clean[table][col], errors="coerce").dt.normalize()
        invalid = int(parsed.isna().sum() - before)
        clean[table][col] = parsed
        type_rows.append([table, col, "date", invalid])

display(pd.DataFrame(type_rows, columns=["table", "column", "target_type", "invalid_after_convert"]))


In [ ]:
# Correct numeric type.
num_cols = {
    "sales": ["Revenue", "COGS"],
    "sample_submission": ["Revenue", "COGS"],
    "web_traffic": ["sessions", "unique_visitors", "page_views", "bounce_rate", "avg_session_duration_sec"],
    "order_items": ["quantity", "unit_price", "discount_amount"],
    "products": ["price", "cogs"],
    "returns": ["return_quantity", "refund_amount"],
    "reviews": ["rating"],
    "promotions": ["discount_value", "min_order_value", "stackable_flag"],
    "inventory": ["stock_on_hand", "units_received", "units_sold", "stockout_days", "days_of_supply", "fill_rate"],
}

for table, cols in num_cols.items():
    if table not in clean:
        continue
    for col in cols:
        if col not in clean[table].columns:
            continue
        before = clean[table][col].isna().sum()
        converted = pd.to_numeric(clean[table][col], errors="coerce")
        invalid = int(converted.isna().sum() - before)
        clean[table][col] = converted
        type_rows.append([table, col, "numeric", invalid])

type_report = pd.DataFrame(type_rows, columns=["table", "column", "target_type", "invalid_after_convert"])
display(type_report.sort_values("invalid_after_convert", ascending=False).head(20))


In [ ]:
# Duplicate handling: chỉ drop exact duplicate, còn duplicate theo key thì báo cáo riêng.
duplicate_rows = []
for name, df in list(clean.items()):
    exact_dup = int(df.duplicated().sum())
    if exact_dup:
        clean[name] = df.drop_duplicates().copy()
    duplicate_rows.append([name, exact_dup, len(df), len(clean[name])])

duplicate_report = pd.DataFrame(duplicate_rows, columns=["table", "exact_duplicate_rows", "rows_before", "rows_after"])
display(duplicate_report.sort_values("exact_duplicate_rows", ascending=False))


In [ ]:
# Missing overview: bảng + biểu đồ top missing để biết cần xử lý ở đâu.
missing_rows = []
for name, df in clean.items():
    miss = df.isna().sum()
    for col, cnt in miss[miss > 0].items():
        missing_rows.append([name, col, int(cnt), round(cnt / len(df) * 100, 2)])

missing_report = pd.DataFrame(missing_rows, columns=["table", "column", "missing_count", "missing_pct"])
display(missing_report.sort_values("missing_pct", ascending=False).head(25))

plot_missing = missing_report.sort_values("missing_count").tail(15).assign(field=lambda d: d.table + "." + d.column)
barh(plot_missing, "missing_count", "field", "Top missing values", color="#E45756", figsize=(8, 5))


In [ ]:
# Logic check: không sửa âm thầm, chỉ báo cáo lỗi/nghi vấn.
logic_rows = []
for table, cols in num_cols.items():
    if table not in clean:
        continue
    for col in cols:
        if col not in clean[table].columns or col in ["bounce_rate", "fill_rate"]:
            continue
        neg = int((clean[table][col] < 0).fillna(False).sum())
        if neg:
            logic_rows.append([table, col, "negative_value", neg, "review_before_fix"] )

if "sales" in clean and has_cols(clean["sales"], ["Revenue", "COGS"]):
    bad_margin = int((clean["sales"]["Revenue"] < clean["sales"]["COGS"] * 0.5).sum())
    logic_rows.append(["sales", "Revenue/COGS", "very_low_margin_day", bad_margin, "review_only"] )

if {"orders", "order_items", "products"}.issubset(clean):
    item_check = clean["order_items"].merge(clean["products"][["product_id", "cogs"]], on="product_id", how="left")
    item_check["gross_line_value"] = item_check["quantity"] * item_check["unit_price"]
    below_cogs = int((item_check["unit_price"] < item_check["cogs"] * 0.95).fillna(False).sum())
    discount_over_gross = int((item_check.get("discount_amount", 0) > item_check["gross_line_value"]).fillna(False).sum())
    logic_rows += [
        ["order_items", "unit_price/cogs", "price_below_cogs", below_cogs, "review_only"],
        ["order_items", "discount/gross", "discount_exceeds_gross", discount_over_gross, "review_only"],
    ]

logic_report = pd.DataFrame(logic_rows, columns=["table", "column", "check", "issue_count", "action"])
display(logic_report)


In [ ]:
# Outlier flag bằng IQR: dùng để soi, không dùng để tự động xóa/cap.
outlier_checks = {
    "sales": ["Revenue", "COGS"],
    "order_items": ["quantity", "unit_price", "discount_amount"],
    "returns": ["refund_amount"],
    "web_traffic": ["sessions", "page_views"],
}

outlier_rows = []
for table, cols in outlier_checks.items():
    if table not in clean:
        continue
    for col in cols:
        if col not in clean[table].columns:
            continue
        s = clean[table][col].dropna()
        if len(s) == 0:
            continue
        lo, hi = iqr_bounds(s)
        cnt = int(((s < lo) | (s > hi)).sum())
        outlier_rows.append([table, col, cnt, round(cnt / len(s) * 100, 2), lo, hi, "flag_only"])

outlier_report = pd.DataFrame(outlier_rows, columns=["table", "column", "outlier_count", "outlier_pct", "iqr_low", "iqr_high", "action"])
display(outlier_report.sort_values("outlier_count", ascending=False))

plot_outlier = outlier_report.assign(field=lambda d: d.table + "." + d.column).sort_values("outlier_count")
barh(plot_outlier, "outlier_count", "field", "IQR outlier count", color="#F58518")


In [ ]:
# Business context cho outlier item-level: giá cao có hợp lệ không thì phải xem COGS/margin.
item_outlier_context = pd.DataFrame()
if {"orders", "order_items", "products"}.issubset(clean):
    items = clean["order_items"].merge(
        clean["orders"][["order_id", "order_date"]], on="order_id", how="left", validate="many_to_one"
    ).merge(
        clean["products"][["product_id", "product_name", "category", "price", "cogs"]], on="product_id", how="left", validate="many_to_one"
    )
    items["gross_line_value"] = items["quantity"] * items["unit_price"]
    items["unit_margin"] = items["unit_price"] - items["cogs"]
    items["margin_rate"] = items["unit_margin"] / items["unit_price"].replace(0, np.nan)
    items["price_to_cogs"] = items["unit_price"] / items["cogs"].replace(0, np.nan)

    _, price_hi = iqr_bounds(items["unit_price"].dropna())
    item_outlier_context = items[items["unit_price"] > price_hi].copy()
    item_outlier_context["business_validity"] = np.select(
        [
            item_outlier_context["cogs"].isna(),
            item_outlier_context["margin_rate"] < -0.05,
            item_outlier_context["price_to_cogs"] > 5,
            item_outlier_context["cogs"] >= items["cogs"].quantile(0.90),
        ],
        ["review_missing_cogs", "possible_error_below_cogs", "review_extreme_markup", "likely_valid_high_cogs"],
        default="review_high_price",
    )
    item_outlier_context = item_outlier_context[[
        "order_id", "order_date", "product_id", "product_name", "category", "quantity",
        "unit_price", "price", "cogs", "margin_rate", "price_to_cogs", "business_validity"
    ]].sort_values("unit_price", ascending=False).head(50)

display(item_outlier_context)


## 2. FEATURE

Mục này tạo bảng daily modeling, sinh feature mới, phân tích nhanh feature-target và chọn feature đủ sạch. Feature cùng ngày từ event source sẽ không đi thẳng vào X; chúng được chuyển thành lag/rolling để tránh leakage.


In [ ]:
# Base daily target từ sales.
sales = clean["sales"][["Date", "Revenue", "COGS"]].dropna(subset=["Date"]).copy()
daily_base = sales.groupby("Date", as_index=False).agg(Revenue=("Revenue", "sum"), COGS=("COGS", "sum")).sort_values("Date")

daily_base["day_of_week"] = daily_base["Date"].dt.dayofweek
daily_base["month"] = daily_base["Date"].dt.month
daily_base["quarter"] = daily_base["Date"].dt.quarter
daily_base["year"] = daily_base["Date"].dt.year
daily_base["is_weekend"] = daily_base["day_of_week"].isin([5, 6]).astype(int)
holidays = USFederalHolidayCalendar().holidays(daily_base["Date"].min(), daily_base["Date"].max()).normalize()
daily_base["is_holiday"] = daily_base["Date"].isin(holidays).astype(int)
daily_base["days_to_year_end"] = (daily_base["Date"].dt.year.map(lambda y: pd.Timestamp(f"{y}-12-31")) - daily_base["Date"]).dt.days

display(daily_base.head())

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(daily_base["Date"], daily_base["Revenue"], label="Revenue", linewidth=1)
ax.plot(daily_base["Date"], daily_base["COGS"], label="COGS", linewidth=1)
ax.set_title("Daily Revenue/COGS")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Traffic features.
feature_tables = {}

if "web_traffic" in clean and "date" in clean["web_traffic"]:
    traffic = clean["web_traffic"]
    traffic_daily = traffic.groupby("date", as_index=False).agg(
        sessions_sum=("sessions", "sum"),
        visitors_sum=("unique_visitors", "sum"),
        page_views_sum=("page_views", "sum"),
        bounce_rate_mean=("bounce_rate", "mean"),
    ).rename(columns={"date": "Date"})

    if "traffic_source" in traffic.columns:
        src_traffic = traffic.pivot_table(index="date", columns="traffic_source", values="sessions", aggfunc="sum", fill_value=0).reset_index()
        src_traffic = src_traffic.rename(columns={"date": "Date"})
        src_traffic.columns = ["Date" if c == "Date" else f"sessions_source_{c}" for c in src_traffic.columns]
        traffic_daily = traffic_daily.merge(src_traffic, on="Date", how="left")

    feature_tables["traffic"] = traffic_daily

show(feature_tables.get("traffic", pd.DataFrame()))


In [ ]:
# Order and item features.
if "orders" in clean and has_cols(clean["orders"], ["order_id", "order_date"]):
    orders = clean["orders"]
    orders_daily = orders.groupby("order_date", as_index=False).agg(
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        order_source_count=("order_source", "nunique"),
    ).rename(columns={"order_date": "Date"})

    status = pd.crosstab(orders["order_date"], orders["order_status"]).reset_index().rename(columns={"order_date": "Date"})
    status.columns = ["Date" if c == "Date" else f"order_status_{c}_count" for c in status.columns]
    feature_tables["orders"] = orders_daily.merge(status, on="Date", how="left")

if {"orders", "order_items"}.issubset(clean):
    items = clean["order_items"].merge(clean["orders"][["order_id", "order_date"]], on="order_id", how="left", validate="many_to_one")
    if "products" in clean and has_cols(clean["products"], ["product_id", "cogs"]):
        items = items.merge(clean["products"][["product_id", "cogs"]], on="product_id", how="left", validate="many_to_one")
    if "discount_amount" not in items:
        items["discount_amount"] = 0
    items["gross_item_revenue"] = items["quantity"] * items["unit_price"]
    items["estimated_cogs"] = items["quantity"] * items.get("cogs", 0)
    feature_tables["items"] = items.groupby("order_date", as_index=False).agg(
        item_quantity_sum=("quantity", "sum"),
        gross_item_revenue_sum=("gross_item_revenue", "sum"),
        discount_amount_sum=("discount_amount", "sum"),
        estimated_cogs_sum=("estimated_cogs", "sum"),
        distinct_product_count=("product_id", "nunique"),
    ).rename(columns={"order_date": "Date"})

show(feature_tables.get("orders", pd.DataFrame()))
show(feature_tables.get("items", pd.DataFrame()))


In [ ]:
# Return features.
if "returns" in clean and "return_date" in clean["returns"]:
    feature_tables["returns"] = clean["returns"].groupby("return_date", as_index=False).agg(
        return_count=("return_id", "count"),
        return_quantity_sum=("return_quantity", "sum"),
        refund_amount_sum=("refund_amount", "sum"),
    ).rename(columns={"return_date": "Date"})

show(feature_tables.get("returns", pd.DataFrame()))


In [ ]:
# Inventory monthly snapshot -> daily forward-fill.
if "inventory" in clean and "snapshot_date" in clean["inventory"]:
    inv = clean["inventory"].groupby("snapshot_date", as_index=False).agg(
        stock_on_hand_sum=("stock_on_hand", "sum"),
        units_sold_sum=("units_sold", "sum"),
        stockout_days_sum=("stockout_days", "sum"),
        fill_rate_mean=("fill_rate", "mean"),
        inventory_product_count=("product_id", "nunique"),
    ).rename(columns={"snapshot_date": "Date"}).sort_values("Date")

    all_dates = pd.date_range(daily_base["Date"].min(), daily_base["Date"].max(), freq="D")
    feature_tables["inventory"] = inv.set_index("Date").reindex(all_dates).ffill().reset_index().rename(columns={"index": "Date"})

show(feature_tables.get("inventory", pd.DataFrame()))


In [ ]:
# Promotion features known before prediction date.
if "promotions" in clean and has_cols(clean["promotions"], ["start_date", "end_date", "discount_value"]):
    promo = clean["promotions"].dropna(subset=["start_date", "end_date"]).copy()
    promo_daily = pd.DataFrame({"Date": pd.date_range(daily_base["Date"].min(), daily_base["Date"].max(), freq="D")})
    rows = []
    for d in promo_daily["Date"]:
        active = promo[(promo["start_date"] <= d) & (promo["end_date"] >= d)]
        rows.append([len(active), active["discount_value"].mean() if len(active) else 0, active["discount_value"].max() if len(active) else 0])
    promo_daily[["promo_count_active", "promo_discount_mean", "promo_discount_max"]] = rows
    promo_daily["is_promo_active"] = (promo_daily["promo_count_active"] > 0).astype(int)
    feature_tables["promotions"] = promo_daily

show(feature_tables.get("promotions", pd.DataFrame()))


In [ ]:
# Review and customer acquisition features.
if "reviews" in clean and has_cols(clean["reviews"], ["review_date", "rating"]):
    feature_tables["reviews"] = clean["reviews"].groupby("review_date", as_index=False).agg(
        review_count=("rating", "count"),
        rating_mean=("rating", "mean"),
        low_rating_count=("rating", lambda x: (x <= 2).sum()),
    ).rename(columns={"review_date": "Date"})

if "customers" in clean and has_cols(clean["customers"], ["customer_id", "signup_date"]):
    feature_tables["customers"] = clean["customers"].groupby("signup_date", as_index=False).agg(
        new_customer_count=("customer_id", "nunique"),
    ).rename(columns={"signup_date": "Date"})

show(feature_tables.get("reviews", pd.DataFrame()))
show(feature_tables.get("customers", pd.DataFrame()))


In [ ]:
# Kiểm tra grain rồi join vào daily base.
agg_report = []
for name, df in feature_tables.items():
    agg_report.append([name, len(df), df["Date"].nunique(), int(df.duplicated("Date").sum())])
agg_report = pd.DataFrame(agg_report, columns=["feature_table", "rows", "unique_dates", "duplicate_dates"])
display(agg_report)

daily_model = daily_base.copy()
join_rows = []
for name, df in feature_tables.items():
    before = len(daily_model)
    cols = [c for c in df.columns if c != "Date"]
    daily_model = daily_model.merge(df, on="Date", how="left", validate="one_to_one")
    unmatched = int(daily_model[cols].isna().all(axis=1).sum()) if cols else 0
    join_rows.append([name, before, len(daily_model), unmatched])

join_report = pd.DataFrame(join_rows, columns=["feature_table", "rows_before", "rows_after", "unmatched_days"])
display(join_report)


In [ ]:
# Sinh feature leakage-safe.
known_now = ["day_of_week", "month", "quarter", "year", "is_weekend", "is_holiday", "days_to_year_end"]
promo_cols = [c for c in ["promo_count_active", "promo_discount_mean", "promo_discount_max", "is_promo_active"] if c in daily_model.columns]
known_now += promo_cols

daily_model, target_lag_cols = add_lag_roll(daily_model, "Date", TARGETS)
raw_same_day_cols = [
    c for c in daily_model.columns
    if c not in ["Date"] + TARGETS + known_now + target_lag_cols
    and pd.api.types.is_numeric_dtype(daily_model[c])
]
daily_model, source_lag_cols = add_lag_roll(daily_model, "Date", raw_same_day_cols)

all_features = known_now + target_lag_cols + source_lag_cols
feature_table = daily_model[["Date"] + TARGETS + all_features].copy()

print("known_now:", len(known_now))
print("target lag/rolling:", len(target_lag_cols))
print("source lag/rolling:", len(source_lag_cols))
print("same-day raw excluded:", len(raw_same_day_cols))


In [ ]:
# Feature analysis và selection rule đơn giản.
feature_quality = []
for col in all_features:
    s = feature_table[col]
    feature_quality.append([col, round(s.isna().mean() * 100, 2), int(s.nunique(dropna=True)), str(s.dtype)])
feature_quality = pd.DataFrame(feature_quality, columns=["feature", "missing_pct", "nunique", "dtype"])

selected_features = feature_quality.query("nunique > 1 and missing_pct < 95")["feature"].tolist()
feature_schema = feature_quality.assign(selected=lambda d: d.feature.isin(selected_features))

# Correlation chỉ để hiểu feature, không tự động chọn theo toàn bộ target để tránh overfit.
corr_rows = []
for target in TARGETS:
    for feature in selected_features:
        pair = feature_table[[feature, target]].dropna()
        if len(pair) >= 30 and pair[feature].nunique() > 1:
            corr_rows.append([feature, target, pair[feature].corr(pair[target]), pair[feature].corr(pair[target], method="spearman")])
corr_report = pd.DataFrame(corr_rows, columns=["feature", "target", "pearson", "spearman"])

display(feature_schema.sort_values(["selected", "missing_pct"], ascending=[True, False]).head(20))
display(corr_report.assign(abs_pearson=lambda d: d.pearson.abs()).sort_values("abs_pearson", ascending=False).head(20))


In [ ]:
# Biểu đồ nhanh: phân phối target và scatter top feature.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(feature_table["Revenue"], bins=50, color="#4C78A8")
axes[0].set_title("Revenue distribution")
axes[1].hist(feature_table["COGS"], bins=50, color="#F58518")
axes[1].set_title("COGS distribution")
plt.tight_layout()
plt.show()

if len(corr_report):
    top_feature = corr_report.assign(abs_pearson=lambda d: d.pearson.abs()).sort_values("abs_pearson", ascending=False).iloc[0]["feature"]
    sample = feature_table[[top_feature, "Revenue"]].dropna()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(sample[top_feature], sample["Revenue"], s=8, alpha=0.35)
    ax.set_title(f"Revenue vs {top_feature}")
    ax.set_xlabel(top_feature)
    ax.set_ylabel("Revenue")
    plt.tight_layout()
    plt.show()


## 3. FORMAT & EXPORT

Mục này biến feature table thành dữ liệu sẵn cho model: split theo thời gian, impute bằng train-only, scale bằng train-only, PCA nếu bật, rồi xuất CSV sạch.


In [ ]:
# Split theo date cutoff rõ ràng. Bỏ giai đoạn đầu vì lag 365d cần đủ lịch sử.
model_data = feature_table[["Date"] + TARGETS + selected_features].dropna(subset=TARGETS).sort_values("Date").copy()
model_data = model_data[model_data["Date"] >= model_data["Date"].min() + pd.Timedelta(days=max(LAGS))]

train = model_data[model_data["Date"] < VALID_START_DATE].copy()
valid = model_data[model_data["Date"] >= VALID_START_DATE].copy()

X_train_raw = train[selected_features].copy()
X_valid_raw = valid[selected_features].copy()
y_train = train[["Date"] + TARGETS].copy()
y_valid = valid[["Date"] + TARGETS].copy()

split_report = pd.DataFrame([
    ["train", len(train), train["Date"].min(), train["Date"].max()],
    ["valid", len(valid), valid["Date"].min(), valid["Date"].max()],
], columns=["split", "rows", "date_min", "date_max"])
display(split_report)


In [ ]:
# Impute train-only: count/event thiếu -> 0, numeric liên tục -> median của train.
impute_rows = []
X_train = X_train_raw.copy()
X_valid = X_valid_raw.copy()

zero_tokens = ["count", "quantity", "sessions", "visitors", "views", "return", "refund", "stock", "discount", "promo"]
for col in selected_features:
    if any(t in col for t in zero_tokens):
        method = "fill_0"
        value = 0
    else:
        method = "train_median"
        value = X_train[col].median()
        if pd.isna(value):
            value = 0

    tr_before = int(X_train[col].isna().sum())
    va_before = int(X_valid[col].isna().sum())
    X_train[col] = X_train[col].fillna(value)
    X_valid[col] = X_valid[col].fillna(value)
    impute_rows.append([col, method, value, tr_before, int(X_train[col].isna().sum()), va_before, int(X_valid[col].isna().sum())])

imputation_report = pd.DataFrame(impute_rows, columns=[
    "feature", "method", "fill_value_from_train", "train_missing_before", "train_missing_after", "valid_missing_before", "valid_missing_after"
])
display(imputation_report.sort_values(["train_missing_before", "valid_missing_before"], ascending=False).head(20))


In [ ]:
# Scale train-only. Bản raw vẫn được export; bản scaled dùng cho linear/PCA.
scale_mean = X_train.mean()
scale_std = X_train.std().replace(0, 1)
X_train_scaled = (X_train - scale_mean) / scale_std
X_valid_scaled = (X_valid - scale_mean) / scale_std

scale_report = pd.DataFrame({
    "feature": selected_features,
    "mean_from_train": scale_mean.values,
    "std_from_train": scale_std.values,
})

high_corr_report = pd.DataFrame()
if X_train_scaled.shape[1] > 1:
    corr = X_train_scaled.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    high_corr_report = upper.stack().reset_index()
    high_corr_report.columns = ["feature_a", "feature_b", "abs_corr"]
    high_corr_report = high_corr_report[high_corr_report["abs_corr"] >= 0.95].sort_values("abs_corr", ascending=False)

display(scale_report.head())
display(high_corr_report.head(20))


In [ ]:
# PCA nếu cần. Mặc định tắt để notebook dễ hiểu; bật APPLY_PCA=True khi dùng model tuyến tính và high_corr nhiều.
pca_report = pd.DataFrame()
X_train_pca = pd.DataFrame(index=X_train.index)
X_valid_pca = pd.DataFrame(index=X_valid.index)

if APPLY_PCA and X_train_scaled.shape[1] > 1:
    U, S, Vt = np.linalg.svd(X_train_scaled.values, full_matrices=False)
    explained = (S ** 2) / np.sum(S ** 2)
    keep = int(np.searchsorted(np.cumsum(explained), 0.95) + 1)

    train_scores = X_train_scaled.values @ Vt[:keep].T
    valid_scores = X_valid_scaled.values @ Vt[:keep].T
    cols = [f"PC{i+1}" for i in range(keep)]
    X_train_pca = pd.DataFrame(train_scores, columns=cols, index=X_train.index)
    X_valid_pca = pd.DataFrame(valid_scores, columns=cols, index=X_valid.index)
    pca_report = pd.DataFrame({"component": cols, "explained_variance_ratio": explained[:keep]})

display(pca_report)


In [ ]:
# Final checks.
provided_test = clean["sample_submission"].drop_duplicates("Date").sort_values("Date") if "sample_submission" in clean else pd.DataFrame()

final_checks = pd.DataFrame([
    ["X_train_no_missing", X_train.isna().sum().sum() == 0],
    ["X_valid_no_missing", X_valid.isna().sum().sum() == 0],
    ["train_before_valid", train["Date"].max() < valid["Date"].min()],
    ["date_not_in_X", "Date" not in X_train.columns],
    ["target_not_in_X", not any(t in X_train.columns for t in TARGETS)],
    ["provided_test_unique_date", provided_test.empty or provided_test["Date"].is_unique],
], columns=["check", "pass"])

display(final_checks)


In [ ]:
# Export artifacts và reports.
objects = {
    "daily_model_base.csv": daily_model,
    "feature_table_daily.csv": feature_table,
    "X_train_raw.csv": X_train,
    "X_valid_raw.csv": X_valid,
    "X_train_scaled.csv": X_train_scaled,
    "X_valid_scaled.csv": X_valid_scaled,
    "y_train.csv": y_train,
    "y_valid.csv": y_valid,
    "provided_test_template.csv": provided_test,
    "feature_schema.csv": feature_schema,
}
if APPLY_PCA:
    objects["X_train_pca.csv"] = X_train_pca
    objects["X_valid_pca.csv"] = X_valid_pca

reports = {
    "load_report.csv": load_report,
    "type_report.csv": type_report,
    "duplicate_report.csv": duplicate_report,
    "missing_report.csv": missing_report,
    "logic_report.csv": logic_report,
    "outlier_report.csv": outlier_report,
    "item_outlier_context.csv": item_outlier_context,
    "aggregation_report.csv": agg_report,
    "join_report.csv": join_report,
    "correlation_report.csv": corr_report,
    "split_report.csv": split_report,
    "imputation_report.csv": imputation_report,
    "scale_report.csv": scale_report,
    "high_corr_report.csv": high_corr_report,
    "pca_report.csv": pca_report,
    "final_checks.csv": final_checks,
}

manifest = []
for file, df in objects.items():
    path = OUT_DIR / file
    df.to_csv(path, index=False)
    manifest.append([file, len(df), df.shape[1], "artifact", str(path)])

for file, df in reports.items():
    path = REPORT_DIR / file
    df.to_csv(path, index=False)
    manifest.append([f"reports/{file}", len(df), df.shape[1], "report", str(path)])


In [ ]:
# Export clean source tables và manifest.
for name, df in clean.items():
    path = CLEAN_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    manifest.append([f"clean_tables/{name}.csv", len(df), df.shape[1], "clean_table", str(path)])

export_manifest = pd.DataFrame(manifest, columns=["file", "rows", "cols", "kind", "path"])
export_manifest.to_csv(OUT_DIR / "export_manifest.csv", index=False)
display(export_manifest)

print("Done")
print("selected features:", len(selected_features))
print("train/valid:", len(train), len(valid))
print("output:", OUT_DIR)
